# 步驟 4：Vectorbt 回測引擎
把推論出來的權重轉換成交易紀錄，模擬完整的每日淨值與最大回撤(Drawdown)。

In [ ]:
import vectorbt as vbt

# Expand monthly weights to daily (forward-fill)
weights_daily = weights_df.reindex(close.index, method="ffill").fillna(0.0)

# Use common clean stocks
close_clean = (close.loc[TEST_START:]
               .replace(0, np.nan)
               .ffill().bfill()
               .dropna(axis=1))
common      = close_clean.columns.intersection(weights_daily.columns)
c_vbt       = close_clean[common]
w_vbt       = weights_daily.loc[TEST_START:, common]

# Benchmark 0050
bm_close    = close_clean["0050"] if "0050" in close_clean.columns else close_clean.iloc[:, 0]

print(f"  Universe : {c_vbt.shape[1]} stocks × {c_vbt.shape[0]} days")
print(f"  Running…")

pf = vbt.Portfolio.from_orders(
    close       = c_vbt,
    size        = w_vbt,
    size_type   = "targetpercent",
    group_by    = True,
    cash_sharing= True,
    fees        = FEE,
    slippage    = 0.001,
    init_cash   = INIT_CASH,
)

pf_bm = vbt.Portfolio.from_holding(bm_close, init_cash=INIT_CASH)

# ─── Print stats ─────────────────────────────────────────────────────────────
stats = pf.stats()
stats_bm  = pf_bm.stats()

COMPARE_KEYS = [
    ("Total Return [%]",         "Total Return"),
    ("Max Drawdown [%]",         "Max Drawdown"),
    ("Max Drawdown Duration",    "DD Duration (days)"),
]

# Also compute annualized return manually from vbt equity
eq      = pf.value()
n_years = (eq.index[-1] - eq.index[0]).days / 365.25
cagr = (eq.iloc[-1] / INIT_CASH) ** (1/n_years) - 1

eq_bm     = pf_bm.value()
cagr_bm   = (eq_bm.iloc[-1] / INIT_CASH) ** (1/n_years) - 1

monthly_eq   = eq.resample("ME").last()
monthly_ret_ = monthly_eq.pct_change().dropna()
sharpe    = (monthly_ret_.mean() / monthly_ret_.std()) * (12**0.5)

monthly_bm_ = eq_bm.resample("ME").last().pct_change().dropna()
sharpe_bm   = (monthly_bm_.mean() / monthly_bm_.std()) * (12**0.5)

print()
print("  ╔══════════════════════════════════════════════════════╗")
print("  ║   Strategy vs Benchmark (0050) — vectorbt        ║")
print("  ╠════════════════════╦═══════════════╦════════════════╣")
print("  ║ Metric             ║  Strategy  ║  Benchmark     ║")
print("  ╠════════════════════╬═══════════════╬════════════════╣")

rows = [
    ("CAGR",           f"{cagr:.2%}",               f"{cagr_bm:.2%}"),
    ("Total Return",   f"{stats['Total Return [%]']:.2f}%",  f"{stats_bm['Total Return [%]']:.2f}%"),
    ("Max Drawdown",   f"{stats['Max Drawdown [%]']:.2f}%",  f"{stats_bm['Max Drawdown [%]']:.2f}%"),
    ("Sharpe (Monthly Ann.)", f"{sharpe:.2f}",        f"{sharpe_bm:.2f}"),
    ("Active Months",  f"{active_months} / {len(weights_df)}", "35 / 35"),
]
for label, s_val, b_val in rows:
    win = "✓" if label in ["CAGR","Total Return","Sharpe (Monthly Ann.)"] and float(s_val.rstrip("% ")) > float(b_val.rstrip("% ")) else \
          "✓" if label == "Max Drawdown" and float(s_val.rstrip("% ")) > float(b_val.rstrip("% ")) else " "
    print(f"  ║ {label:<18} ║ {s_val:>12} {win} ║ {b_val:>13} ║")

print("  ╚════════════════════╩═══════════════╩════════════════╝")

# v5 vs comparison
print()
print("  ── v5 vs comparison (vectorbt, same period) ──")
print(f"  v5  Total Return : 59.97%    Max DD: 22.53%   Sharpe: 0.90")
print(f"   Total Return : {stats['Total Return [%]']:.2f}%    "
      f"Max DD: {stats['Max Drawdown [%]']:.2f}%   "
      f"Sharpe: {sharpe:.2f}")

# Save predictions for report reuse
final_preds.to_pickle("predictions.pkl")
weights_df.to_pickle("../weights.pkl")
eq.to_pickle("eq.pkl")
eq_bm.to_pickle("bm_eq.pkl")
print("\n  📦 Saved: predictions.pkl / weights.pkl / eq.pkl")

print("\n✅ Done!")
import subprocess
subprocess.run(["python3", "generate_report.py"])